In [0]:
%run "./02_data_preparation"

In [0]:
# ============================================================
# 04.3 PARÂMETROS DO PROFILE
# ============================================================

# O QUE FAZ:
# Define os parâmetros específicos utilizados pelas análises do Distribution Profile.

# COMO FAZ:
# Define a quantidade de valores exibidos nas análises Top N e os níveis utilizados para medir a concentração dos valores.

# POR QUE É IMPORTANTE:
# Mantém as configurações específicas do Distribution Profile próximas às análises que utilizam esses parâmetros, sem sobrecarregar o notebook central de bibliotecas.

# PERGUNTA RESPONDIDA:
# "Quais parâmetros serão utilizados nas análises de distribuição?"

TOP_N = 5

TOP_CONCENTRATION_N = [1, 3, 5]

print(f"TOP_N: {TOP_N}")
print(f"TOP_CONCENTRATION_N: {TOP_CONCENTRATION_N}")

In [0]:
# ============================================================
# 04.3 VALIDAÇÃO DO DATAFRAME
# ============================================================

# O QUE FAZ:
# Valida se o DataFrame preparado e as informações necessárias para o Distribution Profile estão disponíveis.

# COMO FAZ:
# Verifica a existência do DataFrame preparado, do total de registros e da lista de colunas elegíveis para análise de distribuição.

# POR QUE É IMPORTANTE:
# Garante que o Profile seja executado sobre uma estrutura válida e evita processamento desnecessário.

# PERGUNTA RESPONDIDA:
# "O DataFrame possui as informações necessárias para executar o Distribution Profile?"

if "df_prepared" not in locals():
    raise ValueError(
        "O dataframe 'df_prepared' não foi disponibilizado pelo Data Preparation"
    )

if "total_registers" not in locals():
    raise ValueError(
        "A variável 'total_registers' não foi disponibilizada pelo Data Preparation"
    )

if "columns_distribuition" not in locals():
    raise ValueError(
        "A variável 'columns_distribuition' não foi disponibilizada pelo Data Preparation"
    )

if total_registers == 0:
    raise ValueError(
            "O dataframe 'df_prepared' não possui registros. Verifique se o Data Preparation foi executado corretamente."
    )

if len(columns_distribution) == 0:
    raise ValueError(
        "A variável 'columns_distribution' não possui colunas elegíveis para análise de distribuição. Verifique se o Data Preparation foi executado corretamente."
    )

print("Dataframe validado com sucesso")
print(f"Total de registros: {total_registers}")
print(f"Colunas analisáveis: {len(columns_distribution)}")


In [0]:
# ============================================================
# 04.5 RESUMO DA DISTRIBUIÇÃO
# ============================================================

# O QUE FAZ:
# Calcula um resumo estatístico das colunas elegíveis para o Distribution Profile.

# COMO FAZ:
# Utiliza as funções estatísticas nativas do Spark para calcular quantidade de valores preenchidos, média, desvio padrão, mínimo e máximo para cada coluna numérica.

# POR QUE É IMPORTANTE:
# Fornece uma visão inicial do comportamento quantitativo dos dados e cria uma base para análises posteriores.

# PERGUNTA RESPONDIDA:
# "Como os valores numéricos estão distribuídos em cada coluna?"

distribution_metrics = []

for column in column_distribution:

    field = next(field for field in df_prepared.schema.fields if field.name == column)

    ref_column = F.col(column)

    if isinstance(field.dataType, (__import__("pyspark").sql.types.NumericType)):
        distribution_metrics.extend([
            F.count(ref_column).alias(f"{column}__filled"),
            F.mean(ref_column).alias(f"{column}__mean"),
            F.stddev(ref_column).alias(f"{column}__stddev"),
            F.min(ref_column).alias(f"{column}__min"),
            F.max(ref_column).alias(f"{column}__max"),
        ])

distribution_resume_row = (
    df_prepared
    .agg(*distribution_metrics)
    .first()
)

distribution_resume = []

for column in column_distribution:
    field = next(field for field in df_prepared.schema.fields if field.name == column)

    if isinstance(field.dataType, (__import__("pyspark").sql.types.NumericType)):
        distribution_resume.append(
            (
                column,
                distribution_resume_row[f"{column}__filled"],
                distribution_resume_row[f"{column}__mean"],
                distribution_resume_row[f"{column}__stddev"],
                distribution_resume_row[f"{column}__min"],
                distribution_resume_row[f"{column}__max"],
            )
        )
distribution_resume = spark.createDataFrame(
    distribution_resume,
    ["column", "filled", "mean", "stddev", "min", "max"]
)

display(distribution_resume)

In [0]:
# ============================================================
# 04.6 FREQUÊNCIA DOS VALORES
# ============================================================

# O QUE FAZ:
# Calcula a frequência absoluta e percentual dos valores presentes em cada coluna analisável.

# COMO FAZ:
# Realiza uma agregação por coluna e valor, calculando a quantidade de ocorrências e o percentual de participação de cada valor no total de registros da respectiva coluna.

# POR QUE É IMPORTANTE:
# Permite identificar valores predominantes, categorias raras, possíveis concentrações e comportamentos assimétricos.

# PERGUNTA RESPONDIDA:
# "Com que frequência cada valor aparece em cada coluna?"

frequencies = []

for column in column_distribution:

    column_frequency = (
        df_prepared.
        groupBy(F.col(column).cast("string").alias("valor"))
        .agg(
            F.count("*").alias("frequency")
        )
        .withColumn("column", F.lit(column))
        .withColumn("value_pct", F.col("frequency") / F.lit(total_registers) *100)
        .select("column", "value", "frequency", "value_pct")
    )

    frequencies.append(column_frequency)

frequencies_values_df = frequencies[0]

for frequency in frequencies[1:]:
    frequency_values_df = frequency_values_df.unionByName(frequency)

display(frequency_values_df.orderBy("column", F.desc("frequency")))

In [0]:
# ============================================================
# 04.7 TOP VALORES POR COLUNA
# ============================================================

# O QUE FAZ:
# Identifica os valores mais frequentes de cada coluna.

# COMO FAZ:
# Utiliza a frequência calculada anteriormente e aplica uma janela particionada por coluna para criar o ranking dos valores mais frequentes.

# POR QUE É IMPORTANTE:
# Facilita a identificação dos valores predominantes sem necessidade de analisar toda a distribuição.

# PERGUNTA RESPONDIDA:
# "Quais são os valores que mais aparecem em cada coluna?"

window_top_valores = (
    Window.partitionBy("column")
    .orderBy(F.desc("frequency"), F.asc("value"))
)

top5_values = (
    frequencies_values_df
    .withColumn("rank", F.row_number().over(window_top_valores))
)
.filter(F.col("rank") <= TOP_N)
.orderBy("column", "rank")

display(top5_values)

In [0]:
# ============================================================
# 04.8 CONCENTRAÇÃO DOS VALORES
# ============================================================

# O QUE FAZ:
# Mede quanto da distribuição de cada coluna está concentrado nos valores mais frequentes.

# COMO FAZ:
# Utiliza o ranking dos valores e calcula a participação acumulada dos Top 1, Top 3 e Top 5 valores.

# POR QUE É IMPORTANTE:
# Uma concentração elevada pode indicar baixa diversidade, forte predominância de categorias ou possível desequilíbrio na distribuição.

# PERGUNTA RESPONDIDA:
# "Qual percentual dos registros está concentrado nos principais valores de cada coluna?"

base_concentration = (
    frequencys_values_df
    .withColumn("rank", F.row_number().over(window_top_valores))
)

agregations_concentration = []

for n in TOP_CONCENTRATION_N:
    agregations_concentration.append(
        F.sum(
            F.when(F.col("rank") <= n, 
            F.col("value_pct")
        )
        .otherwise(0)).alias(f"top_{n}_pct")
    )

concentration_df = (
    base_concentration
    .groupBy("column")
    .agg(*agregations_concentration)
)

display(concentration_df.orderBy(F.desc("concentration_top1")))

In [0]:
# ============================================================
# 04.9 ÍNDICE DE DIVERSIDADE
# ============================================================

# O QUE FAZ:
# Calcula o Índice de Diversidade de Shannon para cada coluna.

# COMO FAZ:
# Utiliza a proporção de ocorrência de cada valor e calcula a entropia de Shannon:
#
# H = - Σ p(x) * log2(p(x))
#
# Quanto maior o valor de H, maior tende a ser a diversidade observada na distribuição.

# POR QUE É IMPORTANTE:
# Permite avaliar a diversidade da distribuição de forma quantitativa, complementando a análise de concentração.

# PERGUNTA RESPONDIDA:
# "Quão diversificados são os valores presentes em cada coluna?"

df_diversity = (
    frequencia_valores_df
    .groupBy("column")
    .agg(
        F.sum(
            -(
                (F.col("frequency") / F.lit(total_registers))
                *
                F.log2(
                    F.col("frequency") / F.lit(total_registers)
                )
            )
        ).alias("shannon_diversity")
    )
)

display(df_diversity.orderBy(F.desc("shannon_diversity")))

In [0]:
# ============================================================
# 04.10 ÍNDICE DE DOMINÂNCIA
# ============================================================

# O QUE FAZ:
# Calcula o percentual de participação do valor mais frequente de cada coluna.

# COMO FAZ:
# Utiliza a concentração Top 1 já calculada anteriormente.

# POR QUE É IMPORTANTE:
# A dominância mostra de forma direta o quanto o valor mais frequente representa da distribuição da coluna.

# PERGUNTA RESPONDIDA:
# "Qual é a participação do valor mais frequente em cada coluna?"

df_dominance = (
    concentration_df
    .select(
        "column",
        F.col("concentration_top1").alias("dominance_index")
    )
)

display(df_dominance.orderBy(F.desc("dominance_index")))

In [0]:
# ============================================================
# 04.11 REGULARIDADE DA DISTRIBUIÇÃO
# ============================================================

# O QUE FAZ:
# Classifica a regularidade da distribuição dos valores de cada coluna com base na concentração do valor predominante.

# COMO FAZ:
# Utiliza o índice de dominância para estabelecer uma classificação interpretativa.

# POR QUE É IMPORTANTE:
# Permite transformar uma métrica estatística em uma leitura mais simples sobre o comportamento da distribuição.

# PERGUNTA RESPONDIDA:
# "A distribuição da coluna é concentrada ou relativamente distribuída entre seus valores?"

df_regularity = (
    df_dominance
    .withColumn(
        "regularity_classification",
        F.when(
            F.col("dominance_index") >= 80, "HIGHLY CONCENTRATED"
        )
        .when(
            F.col("dominance_index") >= 50, "CONCENTRATED"
        )
        .when(
            F.col("dominance_index") >= 20, "MODERATELY DISTRIBUTED"
        )
        .otherwise(
            "SLIGHTLY DISTRIBUTED"
        )
    )
)

display(regularity_classification.orderBy(F.desc("dominance_index")))

In [0]:
# ============================================================
# 04.12 PERFIL CONSOLIDADO DA DISTRIBUIÇÃO
# ============================================================

# O QUE FAZ:
# Consolida as principais métricas calculadas pelo Distribution Profile em um único DataFrame.

# COMO FAZ:
# Realiza joins entre frequência, concentração, diversidade, dominância e regularidade.

# POR QUE É IMPORTANTE:
# Cria uma visão única da distribuição de cada coluna, facilitando análises posteriores e a utilização do resultado pelo Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Qual é o perfil completo da distribuição de cada coluna?"

distribution_profile_df = (
    concentration_df
    .join(
        diversity_df,
        on="column",
        how="left"
    )
    .join(
        regularity_df,
        on="column",
        how="left"
    )
    .orderBy("column")
)

display(distribution_profile_df)

In [0]:
# ============================================================
# 04.13 RESUMO EXECUTIVO DO DISTRIBUTION PROFILE
# ============================================================

# O QUE FAZ:
# Cria uma visão executiva das principais métricas de distribuição de cada coluna.

# COMO FAZ:
# Seleciona as métricas mais relevantes do perfil consolidado e cria uma leitura interpretativa da distribuição.

# POR QUE É IMPORTANTE:
# Facilita a interpretação dos resultados sem necessidade de analisar todas as métricas individualmente.

# PERGUNTA RESPONDIDA:
# "Qual é o comportamento predominante da distribuição de cada coluna?"

distribution_summary_df = (
    distribution_profile_df
    .select(
        "column",
        "top1_concentration",
        "top3_concentration",
        "top5_concentration",
        "shannon_diversity",
        "dominance_index",
        "regularity_classification
    )
    .withColumn(
        "reading",
        F.when(F.col("dominance_index") >= 80, "Highly concentrated distribution")
        .when(F.col("dominance_index") >= 50, "Distribution is concentrated in the main values")
        .when(F.col("dominance_index") >= 5, "Distribution is mostly diverse")
        .otherwise("Distribution is diverse in all values")
    )

    orderBy(F.desc("dominance_index"))
)

display(distribution_summary_df)

In [0]:
# ============================================================
# 04.14 VISUALIZAÇÃO — TOP VALORES
# ============================================================

# O QUE FAZ:
# Prepara os dados para visualização dos principais valores de cada coluna.

# COMO FAZ:
# Utiliza o resultado do ranking Top N calculado anteriormente.

# POR QUE É IMPORTANTE:
# Permite visualizar rapidamente quais valores dominam a distribuição de cada atributo.

# PERGUNTA RESPONDIDA:
# "Quais valores predominam visualmente em cada coluna?"

visualization_top_values_df = (
    top5_frequency
    .select(
        "column",
        "value",
        "frequency",
        "value_pct",
        "rank"
    )
    .orderBy("column", "rank")
)

display(visualization_top_values_df)

In [0]:
# ============================================================
# 04.15 VISUALIZAÇÃO — CONCENTRAÇÃO
# ============================================================

# O QUE FAZ:
# Prepara as métricas de concentração para visualização.

# COMO FAZ:
# Seleciona os percentuais acumulados dos Top 1, Top 3 e Top 5 valores de cada coluna.

# POR QUE É IMPORTANTE:
# Permite comparar visualmente o nível de concentração das diferentes colunas.

# PERGUNTA RESPONDIDA:
# "Quais colunas apresentam maior concentração de valores?"

visualization_concentration_df = (
    concentration_df
    .select(
        "column",
        "top1_concentration",
        "top3_concentration",
        "top5_concentration"
    )
    .orderBy(F.desc("top1_concentration"))
)

display(visualization_concentration_df)

In [0]:
# ============================================================
# 04.16 VISUALIZAÇÃO — DIVERSIDADE X DOMINÂNCIA
# ============================================================

# O QUE FAZ:
# Prepara uma visão comparativa entre diversidade e dominância das colunas.

# COMO FAZ:
# Utiliza o índice de diversidade de Shannon e o índice de dominância calculados anteriormente.

# POR QUE É IMPORTANTE:
# Permite identificar diferentes comportamentos de distribuição, como colunas com alta diversidade e baixa dominância ou baixa diversidade e alta dominância.

# PERGUNTA RESPONDIDA:
# "Como diversidade e dominância se relacionam entre as colunas?"

visualization_diversity_dominance_df = (
    distribution_profile_df
    .select(
        "column",
        "shannon_diversity",
        "dominance_index",
        "regularity_classification
    )
    .orderBy(F.desc("shannon_diversity"))
)

display(visualization_diversity_dominance_df)